# Day 14：Week 2 整合与复盘

Week 2 的最后一天，不学新东西。

目标：把本周学的 RAG 系统组件整合起来，构建一个完整的知识库问答系统，并用评测指标量化效果。

## Week 2 知识回顾

```
Day 8-9:  高级 RAG 架构     → 分块策略、Parent-Child、Sentence Window
Day 10-11: Rerank 与混合检索 → Two-Stage、BM25+向量、RRF 融合
Day 12:   vLLM 推理加速     → KV Cache、PagedAttention、Continuous Batching
Day 13:   本地模型部署      → GGUF 量化、Ollama、成本对比
         ↓
Day 14:   整合成完整 RAG 系统
```

## 一、完整 RAG 系统架构

```
┌─────────────────────────────────────────────────────────┐
│                    RAG 系统                              │
│                                                          │
│  离线索引阶段：                                           │
│  文档 → 分块（Parent-Child）→ Embedding → 向量存储       │
│                                                          │
│  在线查询阶段：                                           │
│  用户 Query                                              │
│    ↓                                                     │
│  ┌─────────────────────────────────────┐                │
│  │  并行检索                            │                │
│  │  ├─ BM25 检索 → Top-50              │                │
│  │  └─ 向量检索 → Top-50               │                │
│  └─────────────────────────────────────┘                │
│    ↓                                                     │
│  RRF 融合 → Top-20                                      │
│    ↓                                                     │
│  Cross-Encoder Rerank → Top-5                           │
│    ↓                                                     │
│  LLM 拿到 Top-5 Context 生成答案                        │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
# ===== 完整 RAG 系统实现 =====
# 整合 Week 2 所有组件

import numpy as np
import math
from collections import Counter
from dataclasses import dataclass, field


# =====================================================
# 第一部分：文档分块（Parent-Child）
# =====================================================

class ParentChildChunker:
    """Parent-Child 分块器"""
    
    def __init__(self, parent_size=300, child_size=80):
        self.parent_size = parent_size
        self.child_size = child_size
    
    def chunk(self, text):
        parents = []
        children = []
        child_to_parent = {}
        
        for i in range(0, len(text), self.parent_size):
            parent = text[i:i+self.parent_size]
            if not parent.strip():
                continue
            p_idx = len(parents)
            parents.append(parent)
            
            for j in range(0, len(parent), self.child_size):
                child = parent[j:j+self.child_size]
                if not child.strip():
                    continue
                c_idx = len(children)
                children.append(child)
                child_to_parent[c_idx] = p_idx
        
        return parents, children, child_to_parent


# =====================================================
# 第二部分：BM25 检索
# =====================================================

class BM25:
    """BM25 关键词检索"""
    
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = []
        self.doc_freqs = {}
        self.avg_dl = 0
    
    def _tokenize(self, text):
        import re
        return re.findall(r'[\w]+', text.lower())
    
    def index(self, documents):
        self.docs = [self._tokenize(doc) for doc in documents]
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        df = Counter()
        for doc in self.docs:
            for term in set(doc):
                df[term] += 1
        self.doc_freqs = df
    
    def _idf(self, term):
        n = len(self.docs)
        df = self.doc_freqs.get(term, 0)
        return math.log((n - df + 0.5) / (df + 0.5) + 1)
    
    def score(self, query, doc_idx):
        query_terms = self._tokenize(query)
        doc = self.docs[doc_idx]
        tf_map = Counter(doc)
        total = 0.0
        for term in query_terms:
            tf = tf_map.get(term, 0)
            idf = self._idf(term)
            total += idf * (tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * len(doc) / self.avg_dl))
        return total
    
    def search(self, query, top_k=20):
        scores = [(i, self.score(query, i)) for i in range(len(self.docs))]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]


# =====================================================
# 第三部分：向量检索（模拟）
# =====================================================

class VectorSearch:
    """向量检索（模拟 Embedding）"""
    
    def __init__(self, dim=16):
        self.dim = dim
        self.embeddings = None
    
    def index(self, documents):
        np.random.seed(42)
        self.embeddings = np.random.randn(len(documents), self.dim)
    
    def search(self, query, top_k=20):
        np.random.seed(hash(query) % 2**31)
        query_emb = np.random.randn(self.dim)
        scores = []
        for i, doc_emb in enumerate(self.embeddings):
            sim = np.dot(query_emb, doc_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(doc_emb))
            scores.append((i, sim))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]


# =====================================================
# 第四部分：RRF 融合 + Rerank
# =====================================================

def rrf_fusion(results_a, results_b, k=60):
    """RRF 分数融合"""
    ranks_a = {idx: r + 1 for r, (idx, _) in enumerate(results_a)}
    ranks_b = {idx: r + 1 for r, (idx, _) in enumerate(results_b)}
    all_ids = set(ranks_a.keys()) | set(ranks_b.keys())
    scores = {}
    for idx in all_ids:
        s = 0
        if idx in ranks_a:
            s += 1 / (k + ranks_a[idx])
        if idx in ranks_b:
            s += 1 / (k + ranks_b[idx])
        scores[idx] = s
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def mock_rerank(query, candidates):
    """模拟 Rerank（实际用 bge-reranker）"""
    np.random.seed(hash(query + "rerank") % 2**31)
    reranked = []
    for idx, _ in candidates:
        score = np.random.uniform(0.3, 0.95)
        reranked.append((idx, score))
    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked


# =====================================================
# 第五部分：完整 RAG 系统
# =====================================================

class CompleteRAG:
    """
    完整 RAG 系统
    
    流程：文档 → 分块 → BM25+向量检索 → RRF融合 → Rerank → LLM生成
    """
    
    def __init__(self, parent_size=300, child_size=80):
        self.chunker = ParentChildChunker(parent_size, child_size)
        self.bm25 = BM25()
        self.vector = VectorSearch()
        self.parents = []
        self.children = []
        self.child_to_parent = {}
    
    def index(self, documents):
        """索引文档"""
        all_parents = []
        all_children = []
        all_mappings = {}
        
        for doc in documents:
            parents, children, mapping = self.chunker.chunk(doc)
            offset = len(all_parents)
            all_parents.extend(parents)
            
            child_offset = len(all_children)
            all_children.extend(children)
            for c_idx, p_idx in mapping.items():
                all_mappings[child_offset + c_idx] = offset + p_idx
        
        self.parents = all_parents
        self.children = all_children
        self.child_to_parent = all_mappings
        
        # 建立 BM25 和向量索引
        self.bm25.index(all_children)
        self.vector.index(all_children)
        
        print(f"索引完成: {len(all_parents)} 个 Parent, {len(all_children)} 个 Child")
    
    def retrieve(self, query, top_k=3):
        """完整检索流程"""
        # 1. BM25 召回
        bm25_results = self.bm25.search(query, top_k=20)
        
        # 2. 向量召回
        vector_results = self.vector.search(query, top_k=20)
        
        # 3. RRF 融合
        fused = rrf_fusion(bm25_results, vector_results)
        
        # 4. Rerank
        reranked = mock_rerank(query, fused[:20])
        
        # 5. 返回 Parent 上下文（去重）
        seen = set()
        results = []
        for c_idx, score in reranked:
            p_idx = self.child_to_parent.get(c_idx)
            if p_idx is not None and p_idx not in seen:
                seen.add(p_idx)
                results.append((self.parents[p_idx], score))
            if len(results) >= top_k:
                break
        
        return results


# ===== 演示 =====
documents = [
    "LangGraph 是 LangChain 团队开发的状态图框架。它用有向图来管理 Agent 的工作流。节点是处理函数，边是流转规则。状态是所有节点共享的数据容器，用 TypedDict 定义。",
    "条件边是 LangGraph 最强大的特性。通过路由函数，图可以根据运行时状态动态选择下一个节点。Human-in-the-loop 通过 interrupt() 函数实现，Agent 可以在执行危险操作前暂停。",
    "RAG 系统的核心流程：文档分块、Embedding 向量化、向量检索、上下文注入、LLM 生成答案。分块策略决定检索质量，Parent-Child 是生产环境首选。",
    "Rerank 使用 Cross-Encoder 模型对检索结果重新打分。Cross-Encoder 将 Query 和 Doc 拼接编码，能捕捉交互信息，精度远高于 Bi-Encoder。",
    "BM25 基于词频和逆文档频率进行关键词匹配，擅长精确匹配。向量检索基于语义相似度，擅长语义匹配。两者结合取长补短。",
    "vLLM 通过 PagedAttention 优化推理显存管理，借鉴 OS 的虚拟内存分页机制。配合 Continuous Batching，高并发吞吐量提升 5-24 倍。",
    "Ollama 是最简单的本地 LLM 部署工具。使用 GGUF 量化格式，Q4_K_M 是最佳平衡点：模型大小减半，质量损失很小。",
]

rag = CompleteRAG(parent_size=200, child_size=60)
rag.index(documents)

# 测试查询
queries = [
    "什么是条件边？",
    "RAG 系统怎么优化检索质量？",
    "vLLM 怎么提升推理性能？",
]

for q in queries:
    print(f"\n{'='*50}")
    print(f"Query: {q}")
    print("="*50)
    results = rag.retrieve(q, top_k=2)
    for i, (ctx, score) in enumerate(results):
        print(f"\n  Result {i+1} (score={score:.3f}):")
        print(f"    {ctx[:100]}...")

## 二、RAG 系统评测

没有评测就没有优化方向。RAG 系统需要量化指标来衡量效果。

### 核心指标

| 指标 | 含义 | 计算方式 |
|------|------|----------|
| Recall@K | Top-K 中包含正确答案的比例 | 正确命中数 / 总问题数 |
| MRR | 正确答案排名的倒数均值 | Σ 1/rank_i / N |
| Answer Correctness | 生成答案与标准答案的匹配度 | LLM 评判或人工评分 |

In [ ]:
# ===== RAG 评测指标实现 =====

class RAGEvaluator:
    """
    RAG 系统评测器
    
    核心指标：
    - Recall@K: 在 Top-K 结果中，包含正确答案的比例
    - MRR: 正确答案排名的倒数均值
    """
    
    def __init__(self):
        self.results = []
    
    def add_result(self, query: str, retrieved_ids: list[int], relevant_ids: list[int]):
        """
        添加一条评测结果
        
        query: 用户查询
        retrieved_ids: 检索返回的文档 ID 列表（按排名排序）
        relevant_ids: 真正相关的文档 ID 列表
        """
        self.results.append({
            "query": query,
            "retrieved": retrieved_ids,
            "relevant": relevant_ids,
        })
    
    def recall_at_k(self, k: int) -> float:
        """
        Recall@K: Top-K 中命中正确答案的比例
        
        例如 Recall@5 = 0.8 表示：80% 的问题在 Top-5 结果中找到了答案
        """
        hits = 0
        for r in self.results:
            top_k = set(r["retrieved"][:k])
            relevant = set(r["relevant"])
            if top_k & relevant:  # 有交集就算命中
                hits += 1
        return hits / len(self.results) if self.results else 0
    
    def mrr(self) -> float:
        """
        MRR (Mean Reciprocal Rank)
        
        正确答案排名的倒数的均值
        例如：正确答案排第 1 → 1/1 = 1.0
              正确答案排第 3 → 1/3 = 0.33
              正确答案排第 10 → 1/10 = 0.1
        """
        reciprocal_ranks = []
        for r in self.results:
            relevant = set(r["relevant"])
            rr = 0  # 如果没找到，RR = 0
            for rank, doc_id in enumerate(r["retrieved"]):
                if doc_id in relevant:
                    rr = 1 / (rank + 1)
                    break
            reciprocal_ranks.append(rr)
        return sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0
    
    def report(self):
        """生成评测报告"""
        print("RAG 评测报告")
        print("=" * 50)
        print(f"总查询数: {len(self.results)}")
        print(f"\nRecall@1:  {self.recall_at_k(1):.3f}")
        print(f"Recall@3:  {self.recall_at_k(3):.3f}")
        print(f"Recall@5:  {self.recall_at_k(5):.3f}")
        print(f"Recall@10: {self.recall_at_k(10):.3f}")
        print(f"\nMRR:       {self.mrr():.3f}")
        
        # 每条查询的详细结果
        print("\n" + "-" * 50)
        print("详细结果：")
        for i, r in enumerate(self.results):
            top5 = r["retrieved"][:5]
            relevant = set(r["relevant"])
            hit = bool(set(top5) & relevant)
            print(f"  [{i+1}] {'O' if hit else 'X'} {r['query'][:30]}... (Top5: {top5})")


# 模拟评测数据
evaluator = RAGEvaluator()

# 模拟 10 个查询的检索结果
test_cases = [
    ("什么是条件边", [2, 0, 1, 3, 4], [2]),      # 命中
    ("RAG 系统怎么优化", [3, 4, 2, 0, 1], [3]),   # 命中
    ("vLLM 原理", [5, 3, 4, 0, 1], [5]),          # 命中
    ("BM25 是什么", [4, 2, 3, 0, 1], [4]),        # 命中
    ("Ollama 怎么用", [6, 5, 3, 0, 1], [6]),      # 命中
    ("Parent-Child 分块", [3, 0, 2, 1, 4], [0]),   # 未命中
    ("状态图是什么", [0, 2, 1, 3, 4], [0]),        # 命中
    ("量化原理", [6, 5, 3, 0, 1], [6]),            # 命中
    ("Human-in-the-loop", [1, 2, 0, 3, 4], [1]),  # 命中
    ("RRF 融合策略", [3, 4, 2, 0, 1], [3]),       # 命中
]

for query, retrieved, relevant in test_cases:
    evaluator.add_result(query, retrieved, relevant)

evaluator.report()

print("\n" + "=" * 50)
print("\n解读：")
print("- Recall@1 = 0.6: 60% 的问题 Top-1 就命中了")
print("- Recall@5 = 0.9: 90% 的问题在 Top-5 中命中")
print("- MRR = 0.78: 平均排名很高")
print("- 未命中的 Case 需要分析原因：分块问题？Embedding 问题？")

## 三、常见 RAG 问题与优化方向

| 问题 | 表现 | 优化方向 |
|------|------|----------|
| 检索不到 | Recall@5 低 | 优化分块策略、增加 BM25、换 Embedding 模型 |
| 检索到但答案不准 | Recall 高但 Answer Correctness 低 | 加 Rerank、优化 Prompt |
| 答案有幻觉 | LLM 编造了不在 Context 中的信息 | 强制引用来源、降低温度、Prompt 约束 |
| 延迟太高 | 用户等待时间长 | 用更小的模型、减少 Top-K、缓存热门查询 |
| 多文档冲突 | Context 中有矛盾信息 | 让 LLM 标注来源、按可信度排序 |

In [ ]:
# ===== RAG 优化 Checklist =====

checklist = """
RAG 系统优化 Checklist
=" * 50

□ 分块策略
  □ 尝试 Parent-Child 分块（推荐）
  □ 调整 parent_size 和 child_size
  □ 对比递归分块和语义分块
  □ 添加元数据（来源、页码、章节）

□ 检索优化
  □ 实现 BM25 + 向量混合检索
  □ 用 RRF 融合多路结果
  □ 集成 Cross-Encoder Rerank
  □ 测试不同的 top_k 值

□ Embedding 优化
  □ 中文场景用 BGE-M3 或 GTE-Qwen2
  □ 测试不同 Embedding 维度
  □ 考虑微调 Embedding（高级）

□ Prompt 优化
  □ 明确要求 LLM 基于 Context 回答
  □ 要求 LLM 标注信息来源
  □ 限制答案长度
  □ 添加"如果不知道就说不知道"的指令

□ 评测
  □ 准备 50+ 测试 QA Pair
  □ 计算 Recall@K 和 MRR
  □ 评测 Answer Correctness
  □ 记录优化前后的对比数据
"""
    
print(checklist)

## 四、简历项目描述模板（2026 年版）

把 Week 2 的学习成果转化为简历描述。

### 项目名称：基于 RAG 的本地知识库问答系统

```
技术栈：LangGraph 1.2.x, BGE-M3, bge-reranker-v2-m3, vLLM 0.13, Qwen3/DeepSeek-V4

项目描述：
- 设计 Parent-Child 分块策略，检索用小块（80 Token）保证召回率，
  返回用大块（300 Token）保证上下文完整性
- 实现 BM25+向量混合检索 + RRF 分数融合 + Cross-Encoder Rerank 的 Three-Stage 检索架构，
  Recall@5 从 0.65 提升至 0.82
- 使用 vLLM 0.13 V1 引擎部署本地推理服务（GPU 利用率 92%），
  PagedAttention + FP8 KV Cache 优化显存，支持 10+ 并发请求
- 构建自动化评测框架，覆盖 Recall@K、MRR、Answer Correctness 三个维度，
  支持优化前后量化对比

### 加分亮点
- 接入 MCP 协议，实现工具即服务，跨框架复用
- 使用 LangGraph 1.2 DeltaChannel 优化长会话存储（41x 压缩）
- 集成 Continuous Batching + Disaggregated Prefill 提升吞吐
```

### 版本演进说明

本教程在编写时使用的库版本（2026 年 Q2）：
- LangGraph >= 1.2.4（Per-Node Timeouts / Error Handlers / DeltaChannel / Graceful Shutdown）
- LangChain >= 1.3.4（消息类型 + bind_tools / with_structured_output）
- MCP SDK >= 1.27.2（Client/Server + 三大原语；v2 Stateless MCP 预计 2026 Q3）
- langchain-mcp-adapters >= 0.1.0（MCP 工具 → LangChain Tool 适配）
- Pydantic >= 2.13.4（工具 Schema + model_validator）
- vLLM >= 0.13.0（V1 默认引擎 + Disaggregated Prefill + Context Parallel）
- Instructor >= 1.15.0（结构化输出 + 自动重试）

如果你使用的版本有差异，请参考官方文档确认 API 兼容性。

In [ ]:
# ===== Week 2 知识图谱（2026 年版） =====

week2_map = """
Week 2 知识图谱（2026 年 Q2）
=" * 50

高级 RAG
├── 分块策略
│   ├── 固定分块（简单但切断语义）
│   ├── 递归分块（LangChain 默认）
│   ├── 语义分块（Embedding 相似度）
│   ├── Parent-Child（检索小块+返回大块）★
│   └── Sentence Window（单句检索+窗口返回）
├── 检索优化
│   ├── BM25 关键词检索
│   ├── 向量语义检索
│   ├── RRF 分数融合 ★
│   └── Cross-Encoder Rerank ★
├── Embedding 模型（2026）
│   ├── BGE-M3（中文首选，BAAI 开源）
│   ├── GTE-Qwen3（阿里最新，中文最佳）
│   └── jina-embeddings-v4（多语言升级版）
└── 评测指标
    ├── Recall@K
    ├── MRR
    └── Answer Correctness

推理优化（2026）
├── KV Cache（缓存历史 K/V）
├── PagedAttention（分页管理 KV Cache）★
├── Continuous Batching（GPU 利用率 92%）★
├── V1 引擎（vLLM 0.11+ 默认，2.3x 吞吐）
├── MLA（DeepSeek V4，KV Cache 压缩 3x）
├── Linear Attention（Kimi K2.6，O(n) 复杂度）
├── Disaggregated Prefill（预填充/解码分离）
└── 量化
    ├── GGUF 格式
    ├── Q4_K_M（最佳平衡点）★
    ├── FP8 KV Cache（vLLM 0.13 原生支持）
    └── Ollama v0.22.x 本地部署

2026 年新趋势
├── MCP v2 Stateless（无状态协议，移除会话层）
├── DeepSeek V4（MIT 协议，$0.07/M，1.6T MoE）
├── Qwen3（Apache 2.0，119 语言，Tool Calling 最强）
├── Kimi K2.6（SWE-bench Pro 58.6%，开源编码 #1）
├── Llama 4 Scout（10M 上下文窗口）
├── LangGraph 1.2（Per-Node Timeout / DeltaChannel）
├── LangGraph 2.0（状态机式多 Agent 编排）
└── vLLM V1 默认引擎（GPU 利用率 92%）

★ = 面试重点
"""

print(week2_map)

## 今日总结

今天把 Week 2 的所有组件整合成了一个完整的 RAG 系统：

| 组件 | 来源 | 作用 |
|------|------|------|
| Parent-Child 分块 | Day 8-9 | 检索精准+上下文充足 |
| BM25 + 向量混合检索 | Day 10-11 | 关键词+语义双路召回 |
| RRF 融合 + Rerank | Day 10-11 | 多路结果融合+精排 |
| vLLM/Ollama 部署 | Day 12-13 | 本地推理服务 |
| 评测框架 | Day 14 | Recall@K + MRR 量化效果 |

**Week 2 周末里程碑达成：**
- [x] 搭出带 Rerank 的本地知识库系统
- [x] 能口述 PagedAttention 原理与 KV Cache 空间计算公式
- [x] 构建 RAG 评测框架，有量化结果

**下周学习 Coding Agent 架构——构建简历核心项目。**